# R21-H216 - The image census and the extraction-tool contest

**Round** R21 (images in documents) - gate hypothesis. **Mode** CPU, read-only over the 27-doc benchmark corpus.

Three extraction arms are contested over the corpus behind the neo4j2 graph, every captured image is classified,
and the classification labels are frozen as the round's reference artifact (`data/processed/image-census-h216.json`)
that H218 / H221 / H222 / H224 consume.

- **Arm 1** pymupdf embedded image-object extraction
- **Arm 2** Docling figure / layout detection (run in `.venv-docling`)
- **Arm 3** page-render vector-diff (draw-command clusters minus embedded images) - captures vector-drawn figures that carry no image object

Deterministic extraction runs here; image classification was done by multimodal inspection of 13 montage
contact-sheets (513 sampled unique images, all 27 docs) and is loaded frozen. Cached arm outputs live under
`tmp/image-census-cache/` so this notebook reproduces the census without re-running the expensive extraction.

## Imports

In [1]:
import json, collections
from pathlib import Path

ROOT = Path('/home/lab/workspace/learning/projects/knowledge-graph-foundry')
CACHE = ROOT / 'tmp/image-census-cache'

## Configuration

In [2]:
# 27-doc benchmark corpus (excludes 'Evox ...' which is not in the neo4j2 graph)
BENCH = sorted(json.loads((ROOT / 'tmp/parser-round-cache/entities.json').read_text())['docnames'])
INFO_BEARING = {'product_photo', 'diagram', 'rendered_table', 'chart'}
DISAGREE_THRESHOLD = 0.20      # clause (a)
FULL_BREADTH = 0.30           # clause (b) full-breadth
REFUTED_NARROW = 0.10         # info-bearing images rare
print('corpus docs:', len(BENCH))

corpus docs: 27


## Data loading - cached three-arm extraction outputs

In [3]:
a13 = json.loads((CACHE / 'arms13.json').read_text())         # Arm1 embedded + Arm3 vector regions
a2  = json.loads((CACHE / 'arm2.json').read_text())            # Arm2 docling pictures + tables
asm = json.loads((CACHE / 'assembled.json').read_text())       # frozen per-doc class profiles + labels
arm1, arm3 = a13['arm1'], a13['arm3']
uniq1, lbl1 = asm['uniq1'], asm['lbl1']
CLASS_PROFILE = asm['class_profile']
print('Arm1 placements:', len(arm1), 'unique:', len(uniq1))
print('Arm3 vector regions:', len(arm3))
print('Arm2 docling docs cached:', len(a2), '/ 27')

Arm1 placements: 91175 unique: 1566
Arm3 vector regions: 558
Arm2 docling docs cached: 26 / 27


## Arm inventories and clause (a) - inventory disagreement

In [4]:
A1 = len(uniq1)
A2p = sum(v.get('n_pictures', 0) for v in a2.values())
A2t = sum(v.get('n_tables', 0) for v in a2.values()); A2 = A2p + A2t
A3 = len([r for r in arm3 if r['area_frac'] >= 0.02])
disagree = {'A1_vs_A2': abs(A1-A2)/max(A1,A2), 'A1_vs_A3': abs(A1-A3)/max(A1,A3), 'A2_vs_A3': abs(A2-A3)/max(A2,A3)}
clause_a_pass = all(v >= DISAGREE_THRESHOLD for v in disagree.values())
print(f'Arm1 pymupdf embedded : {A1} unique ({len(arm1)} placements)')
print(f'Arm2 docling          : {A2p} pictures + {A2t} tables = {A2}  (26/27 docs; catalogue OOM)')
print(f'Arm3 vector-render    : {A3} regions')
for k, v in disagree.items(): print(f'  {k} count-disagreement = {v:.1%}')
print('clause (a) PASS:', clause_a_pass, '(>= 20% + Docling dominates info-bearing coverage: raster figures + 226 tables the embedded arm misses)')

Arm1 pymupdf embedded : 1566 unique (91175 placements)
Arm2 docling          : 745 pictures + 226 tables = 971  (26/27 docs; catalogue OOM)
Arm3 vector-render    : 558 regions
  A1_vs_A2 count-disagreement = 38.0%
  A1_vs_A3 count-disagreement = 64.4%
  A2_vs_A3 count-disagreement = 42.5%
clause (a) PASS: True (>= 20% + Docling dominates info-bearing coverage: raster figures + 226 tables the embedded arm misses)


### Content disjointness

The disagreement is not only in counts - the arms capture materially different content. Arm 1 (raster embedded
objects) captures **zero** of the vector-drawn tables / schematics / charts; Arm 3 captures 558 such regions and
none of the raster photos; Arm 2 (Docling) overlaps ~89% of raster figures with the embedded arm at page level but
uniquely adds 226 structured tables. No arm is a superset - Docling is the broadest single information-bearing net.

In [5]:
cls_arm1 = collections.Counter(lbl1)
print('Arm1 unique class distribution:', dict(cls_arm1))
ib1 = sum(1 for l in lbl1 if l in INFO_BEARING)
print(f'Arm1 information-bearing share: {ib1}/{A1} = {ib1/A1:.1%}')
print('docling pictures overlapping an embedded object (same page):',
      asm['a2_match']['page_overlap_with_embedded'], '/', asm['a2_match']['docling_pictures'])

Arm1 unique class distribution: {'product_photo': 1022, 'decorative': 281, 'diagram': 228, 'chart': 26, 'logo_decorative': 9}
Arm1 information-bearing share: 1276/1566 = 81.5%
docling pictures overlapping an embedded object (same page): 662 / 745


## Classification (frozen) and clause (b) - text NOT in the text layer

Classes: `product_photo` / `diagram` / `rendered_table` / `chart` / `logo_decorative` / `decorative` (+ vector regions).
Every doc carries at least one information-bearing image. The clause-(b) question is narrower: does the doc carry an
information-bearing image whose **text is not in the text layer**. Verified by comparing visible text against the
pymupdf text layer.

In [6]:
PIXEL_TEXT_DOCS = {
 'PDF RESmart Service Manual CPAP.pdf': "flowchart node text ('Power Supply','Wrong Version') pixel-only",
 'DreamStation_CPAP_User_Manual.pdf': "device-display screenshot values ('5.5 cmH2O') pixel-only",
 'SleepStyle_200_Operating_Manual.pdf': "regulatory symbol 'IPX1' pixel-only",
 'Brochure_BMC_GIII_A20_Oxygenium_Medical.pdf': "device screen 'Usage Summary'/iCode pixel-only",
 'ARTP_Standards_of_Care_-_CPAP_Devices_(Technical_and_Performance)_Version_5.0_-_05-02-2022.pdf': "schematic dimension '40 mm' + chart labels pixel-only",
 'ResMed-Airsense-11-Manual.pdf': 'touchscreen UI values pixel-only (specs are live text)',
 'Resvent-iBreeze-Auto-CPAP-User-Manual.pdf': 'device-UI screenshot values pixel-only (specs are live text)',
}
n = len(BENCH)
info_bearing_share = 1.0                          # 27/27 docs verified via montage inspection
pixel_text_share = len(PIXEL_TEXT_DOCS) / n
print(f'docs with information-bearing image : 27/27 = {info_bearing_share:.0%}')
print(f'docs with pixel-only info-bearing text: {len(PIXEL_TEXT_DOCS)}/27 = {pixel_text_share:.0%}')
print('reaches full breadth (>=30%):', pixel_text_share >= FULL_BREADTH)
print('refuted-narrow (info-bearing images <10% of docs):', info_bearing_share < REFUTED_NARROW)

docs with information-bearing image : 27/27 = 100%
docs with pixel-only info-bearing text: 7/27 = 26%
reaches full breadth (>=30%): False
refuted-narrow (info-bearing images <10% of docs): False


## Frozen census artifact + machine-readable report

In [7]:
import runpy, sys
# regenerate the frozen artifact + report deterministically from cache
sys.argv = ['build']
g = runpy.run_path(str(ROOT / 'notebooks/_build_census_artifacts.py'))
print('artifact + reports written (see paths above)')

WROTE:
  data/processed/image-census-h216.json  records: 2124
  reports/image-census-h216-20260707T200830Z.json
  reports/pixel-forensics-h217-20260707T200830Z.json
H216 clause(a) disagreement: {'A1_vs_A2': '38.0%', 'A1_vs_A3': '64.4%', 'A2_vs_A3': '42.5%'} -> PASS
H216 clause(b): info-bearing docs 100%, pixel-only-text docs 26% (<30% narrow)
H217: {'text-layer-present': 20, 'absent-entirely': 12}  pixel-only 0/32 = 0% -> REFUTED
STAMP 20260707T200830Z
artifact + reports written (see paths above)


## Verdict

**CONFIRMED-NARROW.** Census complete, labels frozen (`data/processed/image-census-h216.json`).

- **Clause (a) PASS** - three-arm inventories materially disagree (38-64% count disagreement >> 20%); Docling dominates
  information-bearing coverage (raster figures + 226 structured tables the embedded arm misses entirely); the embedded
  arm inflates raw count (1566 unique from 91175 placements, 281 decorative) yet captures zero vector-drawn content;
  the vector arm uniquely captures 558 vector tables / schematics / charts
- **Clause (b) NARROW** - information-bearing images are common (27/27 docs) but images whose *text is not in the text
  layer* are ~26% of docs, below the 30% full-breadth threshold; most rendered labels / captions / specs are
  dual-encoded (present in the text layer). Not refuted-narrow
- **Consequence** - the round narrows to the classes actually present (product photos, diagrams / schematics,
  rendered / vector tables), justified by NEW-content capture, **not** by hidden-text recovery (H217: 0/32 absent
  golds are pixel-only)

**Deviation** - Docling crashed repeatedly on `product_and_solutions_catalog.pdf` (90+ pages, memory); 26/27 docs
have Docling coverage; that doc remains covered by Arm 1 and Arm 3.